## 0. Baselines


In [ ]:
import re
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

from apps.intelligence.ml_split import make_split

df_base = pd.DataFrame(Job.objects.values("id", "title", "description", "seniority", "normalised_title"))
df_base["company_name"] = pd.DataFrame(Job.objects.values("company__name"))["company__name"]
df_base["text"] = df_base["title"] + " " + df_base["description"]

X_tr, X_te, y_tr, y_te = make_split(df_base)

# Majority class
dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
print("=== Baseline 1: Majority class ===")
print(classification_report(y_te, dummy.predict(X_te), zero_division=0))

# Keyword rule
RULES = [
    ("senior_plus", re.compile(r"\b(lead|principal|staff|head|director|manager|vp)\b", re.I)),
    ("junior",      re.compile(r"\b(junior|jr|graduate|intern|entry.?level|trainee)\b", re.I)),
    ("senior",      re.compile(r"\b(senior|sr)\b", re.I)),
]
def kw(title):
    for label, pat in RULES:
        if pat.search(title): return label
    return "senior"

y_kw = df_base.loc[X_te.index, "title"].map(kw)
print("=== Baseline 2: Keyword rule ===")
print(classification_report(y_te, y_kw, zero_division=0))


### Baseline results (test set n=120)

| Model | Accuracy | Macro F1 | junior F1 | senior F1 | senior_plus F1 |
|---|---|---|---|---|---|
| Majority class | 0.40 | 0.19 | 0.00 | 0.00 | 0.57 |
| Keyword rule (title) | 0.52 | 0.41 | 0.06 | 0.60 | 0.56 |

The keyword rule is the meaningful baseline to beat — **macro F1 0.41**.


# Seniority Classifier

In [ ]:
import sys, os, django
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings.local')
django.setup()

## 1. Load & merge classes

In [ ]:
import pandas as pd
from apps.intelligence.models import Job

MERGE = {
    'junior': 'junior', 'mid': 'junior', 'intern': 'junior',
    'senior': 'senior',
    'lead': 'senior_plus', 'principal': 'senior_plus', 'manager': 'senior_plus',
}

df = pd.DataFrame(Job.objects.values('id', 'title', 'description', 'seniority'))
df['label'] = df['seniority'].map(MERGE)
df = df.dropna(subset=['label'])          # drop not_stated
df['text'] = df['title'] + ' ' + df['description']

print(df['label'].value_counts())
print('Total training rows:', len(df))

**What to check:** all 3 classes present, total ~304. If any class is missing, the MERGE map has a typo.

## 2. Train / test split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'],
    test_size=0.2,
    stratify=df['label'],   # keeps class proportions in both splits
    random_state=42
)
print('Train:', len(X_train), '| Test:', len(X_test))
print(y_test.value_counts())

**What to check:** test set has examples of all 3 classes. If `junior` has only 11 test rows, that's expected — just be aware macro-F1 will be noisy for that class.

## 3. Train pipeline (TF-IDF + Logistic Regression)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])

pipe.fit(X_train, y_train)
print('Training done.')

**Why these choices:**
- `sublinear_tf=True` — dampens very frequent words (common in job descriptions)
- `ngram_range=(1,2)` — captures phrases like 'senior engineer'
- `class_weight='balanced'` — compensates for the junior/senior_plus imbalance automatically

## 4. Evaluate

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

y_pred = pipe.predict(X_test)
print(classification_report(y_test, y_pred))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
import matplotlib.pyplot as plt
plt.tight_layout()

**What to look for:**
- Macro F1 above 0.75 → good enough to use
- `junior` recall low → expected with few examples; not a blocker
- `senior` and `senior_plus` confused with each other → the boundary is genuinely fuzzy; acceptable
- Any class with precision AND recall both below 0.5 → that class is not learnable from this data; merge or drop it

## 5. Cross-validate (more reliable than a single split)

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, df['text'], df['label'], cv=cv, scoring='f1_macro')
print(f'CV macro-F1: {scores.mean():.3f} ± {scores.std():.3f}')

**What to look for:**
- High std (>0.08) → model is unstable; your dataset is too small or class merge needs revisiting
- CV score much lower than the single split score → you got a lucky split; trust CV more

## 6. Save the model

In [ ]:
import joblib, pathlib

out = pathlib.Path('../apps/intelligence/ml')
out.mkdir(exist_ok=True)
joblib.dump(pipe, out / 'seniority_classifier.joblib')
print('Saved to', out / 'seniority_classifier.joblib')